# Ops1 — Index

1. [Filtering - select, Normalization & Conditional Columns (CASE WHEN / F.when)](#command/ec10554e-e7c5-48b8-854b-62d172aa3f60)
2. [Standard distinct, Aggregations, Conditional GROUP BY & FILTER (WHERE)](#command/576e0ec3-853b-4238-bc7c-24237b5b49e1)
3. [ROLLUP, CUBE, GROUPING SETS & Advanced Aggregates](#command/793c3a24-872b-4222-8b2f-9616da514d1d)
4. [Join Types & Anti-Joins](#command/63d277f9-a0ff-4534-87ad-27c9d47eb701)

In [0]:
from pyspark.sql import functions as F

### Filtering - select, Normalization & Conditional Columns (CASE WHEN / F.when)

In [0]:
sales_df = spark.table("sql_master.sql_master_pack.sales")

print(f"sales_df columns: {', '.join(sales_df.columns)}")

print("sales_df schema:")
sales_df.printSchema()

row_count = sales_df.count()
print(f"sales_df has {row_count:,} rows")       #:, is for readability - sales_df has 1,500,000 rows

sales_df.limit(4).display()

In [0]:
classified = sales_df.select(
    "sale_id",
    "amount",
    F.when(F.col("amount") < 100, "small")
     .when(F.col("amount") <= 200, "medium")
     .otherwise("large")
     .alias("sale_size")   
).persist()  # Cache in memory/disk to prevent re-computing on the next action
#classified.show()

classified.display()

# Always unpersist when finished
classified.unpersist()

### Standard distinct, Aggregations, Conditional GROUP BY & FILTER (WHERE)

In [0]:
unique_products_df = sales_df.select("product_id").distinct()
unique_products_df.show()

#deduped_df = sales_df.dropDuplicates(["product_id"])

In [0]:
sales_df.groupBy("product_id").agg(
    F.sum("amount").alias("total_amount"),
    F.avg("amount").alias("avg_amount"),
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"),
    F.count("*").alias("transaction_count"),
).show()

### ROLLUP, CUBE, GROUPING SETS & Advanced Aggregates

In [0]:
# Rollup + GROUPING()
sales_df.rollup("region", "product_id").agg(
    F.sum("amount").alias("total"),
    F.grouping("region").alias("region_is_total"),
    F.grouping("product_id").alias("product_is_total"),
).orderBy("region", "product_id").show()

# Cube + Advanced Aggregates
sales_df.cube("region", "product_id").agg(F.sum("amount").alias("total")).show()

sales_df.groupBy("region").agg(
    F.collect_set("product_id").alias("products"),
    F.count_distinct("product_id").alias("n_products"),
    F.approx_count_distinct("sale_id", rsd=0.02).alias("approx_sales"),
    F.percentile_approx("amount", [0.5, 0.9], 10000).alias("p50_p90"),
    F.first("amount", ignorenulls=True).alias("first_amount"),
).show()

### Join Types & Anti-Joins

| Join Type | Keeps |
| :--- | :--- |
| `inner` | Matching rows from both sides |
| `left` / `right` / `full` | All left / right / both sides |
| `left_semi` | Left rows with a match (no right columns) |
| `left_anti` | Left rows with no match (cleaner & safer than `NOT IN`) |

In [0]:
cust_df = spark.table("sql_master.sql_master_pack.customers")
ord_df = spark.table("sql_master.sql_master_pack.orders")

print(f"cust_df columns: {', '.join(cust_df.columns)}")

unique_cust_df = cust_df.select("cust_id").distinct().orderBy("cust_id")

unique_cust_df.display()

In [0]:


inner_j = cust_df.join(ord_df, "cust_id", "inner") # Keeps only matching rows between both DataFrames
left_j  = cust_df.join(ord_df, "cust_id", "left")  # Keeps all rows from cust_df and matching rows from ord_df
left_sj  = cust_df.join(ord_df, "cust_id", "left_semi")  #Keeps rows from cust_df that have AT LEAST ONE matching order in ord_df.
anti_j  = cust_df.join(ord_df, "cust_id", "left_anti") # Customers with 0 orders   Keeps rows from cust_df that have NO matching order in ord_df

full_j = cust_df.join(ord_df, "cust_id", "full")   # Keeps all customers and all orders, with NULL fills where either is missing (useful for reconciliation)

right_j = cust_df.join(ord_df, "cust_id", "right") # Keeps all orders and matching customers (identifies orphaned orders placed with non-existent cust_ids).

inner_j = ord_df.join(F.broadcast(cust_df), "cust_id", "inner") # If one of the tables is small (e.g., cust_df is a small dimension table or lookup under ~10MB–100MB), you should advise applying a broadcast hint to avoid an expensive network shuffle (SortMergeJoin / ShuffleHashJoin):


# Join Fan-Out Check (Always validate dimension key uniqueness before joining!)
dupes = cust_df.groupBy("cust_id").count().filter(F.col("count") > 1)
assert dupes.count() == 0, "Dimension key is not unique!"

In [0]:
left_sj.display()

In [0]:
anti_j.display()

### Recursive and other ops

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

emp_df = spark.table("sql_master.sql_master_pack.employees")

print(f"emp_df columns: {', '.join(emp_df.columns)}")

row_count = emp_df.count()
print(f"emp_df has {row_count:,} rows")       #:, is for readability - sales_df has 1,500,000 rows

In [0]:
# SELECT first_name, last_name, salary
# FROM employees
# WHERE salary > (SELECT AVG(salary) FROM employees);

# avg_emp_sal_df = emp_df.select(F.avg("salary")).first()[0]

# above_company_avg = emp_df.filter(F.col("salary") > avg_emp_sal_df).select ("first_name", "last_name", "salary")
# above_company_avg.show()


above_avg = (
    emp_df
    .withColumn("avg_emp_sal", F.avg("salary").over(Window.partitionBy()))
    .filter(F.col("salary") > F.col("avg_emp_sal"))
    .select("first_name", "last_name", "salary")
)

above_avg.show()

In [0]:
# SELECT e1.first_name, e1.last_name, e1.dept_id, e1.salary
# FROM employees e1
# WHERE e1.salary > (
#     SELECT AVG(e2.salary) FROM employees e2 WHERE e2.dept_id = e1.dept_id
# );

#dept_window = Window.partitionBy("dept_id")

above_dept_avg = (
    emp_df
    .withColumn("dept_avg", F.avg("salary").over(Window.partitionBy("dept_id")))
    .filter(F.col("salary") > F.col("dept_avg"))
    .select("first_name", "last_name", "dept_id", "salary","dept_avg")
)

above_dept_avg.show()

In [0]:
# SELECT e1.first_name, e1.last_name, e1.dept_id
# FROM employees e1
# WHERE EXISTS (
#     SELECT 1 FROM employees e2 WHERE e2.manager_id = 110 AND e2.dept_id = e1.dept_id
# );

# The direct mathematical equivalent of WHERE EXISTS in PySpark is a left_semi join. It filters the left DataFrame against matches in the right DataFrame without duplicating rows or appending right-side columns from the right DataFrame

# Target departments managed by employee 110
mgr_depts = (
    emp_df
    .filter(F.col("manager_id") == 110)
    .select("dept_id")
    .distinct()
)

# left_semi acts as WHERE EXISTS
exists_df = (
    emp_df.alias("e1")
    .join(mgr_depts, on="dept_id", how="left_semi")
    .select("first_name", "last_name", "dept_id")
)

exists_df.show()

In [0]:
subordinates_df = spark.sql("""
    WITH RECURSIVE subordinate AS (
        SELECT employee_id, first_name, last_name, manager_id,
               0 AS hierarchy_level,
               CONCAT(first_name, ' ', last_name) AS reporting_path
        FROM sql_master.sql_master_pack.employees
        WHERE employee_id = 110
        UNION ALL
        SELECT e.employee_id, e.first_name, e.last_name, e.manager_id,
               s.hierarchy_level + 1,
               CONCAT(s.reporting_path, ' -> ', e.first_name, ' ', e.last_name)
        FROM sql_master.sql_master_pack.employees e
        JOIN subordinate s ON e.manager_id = s.employee_id
    )
    SELECT * FROM subordinate ORDER BY hierarchy_level, employee_id
""")

subordinates_df.display()

In [0]:
# 1. Base / Anchor Case (Level 0)
current_level = (
    emp_df
    .filter(F.col("employee_id") == 110)
    .select(
        "employee_id",
        "first_name",
        "last_name",
        "manager_id",
        F.lit(0).alias("hierarchy_level"),
        F.concat_ws(" ", "first_name", "last_name").alias("reporting_path")
    )
)

accumulated = current_level

# 2. Iterative recursive loop
while True:
    next_level = (
        emp_df.alias("e")
        .join(current_level.alias("s"), F.col("e.manager_id") == F.col("s.employee_id"), "inner")
        .select(
            F.col("e.employee_id"),
            F.col("e.first_name"),
            F.col("e.last_name"),
            F.col("e.manager_id"),
            (F.col("s.hierarchy_level") + 1).alias("hierarchy_level"),
            F.concat(
                F.col("s.reporting_path"),
                F.lit(" -> "),
                F.concat_ws(" ", F.col("e.first_name"), F.col("e.last_name"))
            ).alias("reporting_path")
        )
    )

    # Stop when no deeper subordinates exist
    if next_level.isEmpty():
        break

    # Checkpoint/persist can be used here if the hierarchy is deeply nested (> 10 levels)
    accumulated = accumulated.unionByName(next_level)
    current_level = next_level

result_org_chart = accumulated.orderBy("hierarchy_level", "employee_id")

result_org_chart.display()
